# 1P parameter sweep: is feedback really invisible, or just swamped?

In the LH fixed-N result (notebook 02), every simulation varies all 6
parameters at once, so each parameter's quartile-contrast response is
marginalized over the other 5 -- adding noise on top of any real-but-small
feedback signal. The **1P set** varies one parameter at a time with the
others held at fiducial, removing that marginalization noise entirely.

**This particular 1P set turns out to vary 28 astrophysics parameters
(`WindEnergyIn1e51erg`, `RadioFeedbackFactor`, `BlackHoleFeedbackFactor`,
...), not the LH run's 4 lumped ones (`A_SN1`/`A_AGN1`/`A_SN2`/`A_AGN2`)** --
confirmed empirically in section 1 below, not assumed. Only the 2
cosmological columns (`Omega0`/`sigma8`) correspond directly to
`Omega_m`/`sigma_8` and are checked against LH's dominant signal as a sanity
check; the 28 astrophysics columns are a finer decomposition of feedback
physics with no 1:1 match to LH's 4, so they're analyzed and ranked on their
own terms -- "does *any* of this richer decomposition show a real trend",
not a literal re-test of `A_SN1` etc.

**Nothing about the 1P set's directory naming, grid size, step spacing, or
which column each varied index corresponds to is assumed.** Section 1
discovers it from what's actually on disk and the 1P parameter table;
section 2 verifies the index -> column mapping empirically before trusting
it.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src import config
from src.data_io import find_snapshots
from src.params import load_1p_params
from src.onep import (
    parse_1p_label, infer_1p_parameter_names, mean_cdf_by_step, monotonic_trend,
    COSMO_ALIASES,
)
from src.pipeline import run_suite
from src.sensitivity import infer_layout, benjamini_hochberg
from src.plotting import plot_1p_sweep

SIM_PATH_1P = config.SIM_PATH_1P
PARAMS_FILE_1P = config.PARAMS_FILE_1P
OUTPUT_DIR = config.OUTPUT_DIR

## 1. What's actually on disk?

Lists every 1P snapshot found, how many parameters are varied and with what
step set, and the parameter table's shape -- before assuming any of it.

In [2]:
files_1p = find_snapshots(SIM_PATH_1P, config.SNAP, prefix="1P_")
print(f"Found {len(files_1p)} 1P snapshots at snap={config.SNAP}")

labels = sorted({f.split("1P_")[1].split("/")[0] for f in files_1p})
parsed = {l: parse_1p_label(l) for l in labels}

n_fiducial = sum(1 for p, _ in parsed.values() if p is None)
param_steps = {}
for p, s in parsed.values():
    if p is None:
        continue
    param_steps.setdefault(p, []).append(s)

steps_per_param = pd.Series({p: sorted(s) for p, s in param_steps.items()}).sort_index()

print(f"\nShared fiducial ('0') snapshots: {n_fiducial}")
print(f"Parameters varied: {len(steps_per_param)}")
print("\nStep count per parameter:")
print(steps_per_param.apply(len).value_counts().sort_index())

step_sets = steps_per_param.apply(tuple)
modal = step_sets.value_counts().idxmax()
irregular = step_sets[step_sets != modal]

print(f"\nModal step set: {modal}")
if len(irregular):
    print("Parameters with a DIFFERENT step set (worth a manual look):")
    print(irregular)
else:
    print("Every parameter has the same step set.")

theta_1p_all = load_1p_params(PARAMS_FILE_1P)
print(f"\nParameter table: {theta_1p_all.shape[0]} rows, columns = {list(theta_1p_all.columns)}")

Found 113 1P snapshots at snap=50

Shared fiducial ('0') snapshots: 1
Parameters varied: 28

Step count per parameter:
4    28
Name: count, dtype: int64

Modal step set: (-2, -1, 1, 2)
Parameters with a DIFFERENT step set (worth a manual look):
15    (1, 2, 3, 4)
dtype: object

Parameter table: 140 rows, columns = ['#Name', 'Omega0', 'sigma8', 'WindEnergyIn1e51erg', 'RadioFeedbackFactor', 'VariableWindVelFactor', 'RadioFeedbackReiorientationFactor', 'OmegaBaryon', 'HubbleParam', 'n_s', 'MaxSfrTimescale', 'FactorForSofterEQS', 'IMFslope', 'SNII_MinMass_Msun', 'ThermalWindFraction', 'VariableWindSpecMomentum', 'WindFreeTravelDensFac', 'MinWindVel', 'WindEnergyReductionFactor', 'WindEnergyReductionMetallicity', 'WindEnergyReductionExponent', 'WindDumpFactor', 'SeedBlackHoleMass', 'BlackHoleAccretionFactor', 'BlackHoleEddingtonFactor', 'BlackHoleFeedbackFactor', 'BlackHoleRadiativeEfficiency', 'QuasarThreshold', 'QuasarThresholdPower', 'seed']


## 2. Verify the parameter-index -> column mapping

Checks which single column of the parameter table actually varies within
each index group, discovered from the table's own columns -- not assumed to
be `config.ALL_PARAMS` (the LH set's 6 names), since this 1P set's own
parameterization can be, and here is, entirely different. Splits the result
into the 2 columns directly comparable to LH (`Omega_m`/`sigma_8`) and the
rest, this suite's own astrophysics decomposition. Any group in `ambiguous`
needs a manual look before proceeding.

In [3]:
mapping, ambiguous = infer_1p_parameter_names(theta_1p_all)

print(f"Resolved {len(mapping)} of {len(mapping) + len(ambiguous)} groups cleanly.")
if ambiguous:
    print("AMBIGUOUS groups (inspect by hand):", ambiguous)
assert not ambiguous, "Resolve ambiguous groups before trusting the mapping below."

cosmo_mapping = {pidx: name for pidx, name in mapping.items() if name in COSMO_ALIASES}
astro_mapping = {pidx: name for pidx, name in mapping.items() if name not in COSMO_ALIASES}

print(f"\nCosmological ({len(cosmo_mapping)}, directly comparable to LH):")
for pidx, name in sorted(cosmo_mapping.items()):
    print(f"  index {pidx}: {name}  (LH: {COSMO_ALIASES[name]})")

print(f"\nAstrophysics ({len(astro_mapping)}, this suite's own decomposition -- "
      f"not a 1:1 match to LH's A_SN1/A_AGN1/A_SN2/A_AGN2):")
for pidx, name in sorted(astro_mapping.items()):
    print(f"  index {pidx}: {name}")

TypeError: unsupported operand type(s) for -: 'str' and 'str'

## 3. Generate fixed-N AGN summaries over the 1P set

Same `N_TARGET`/`mass_cut`/`kvals`/`rgrid` as the LH fixed-N run (notebook
02), read from its saved `.npz`, so results are directly comparable.

In [ ]:
import glob

agn_candidates = sorted(glob.glob(f"{OUTPUT_DIR}/agn_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n*.npz"))
assert agn_candidates, f"No LH AGN fixed-N run found in {OUTPUT_DIR} -- run notebook 02 first."
agn_data = np.load(agn_candidates[-1], allow_pickle=True)
N_TARGET = int(agn_data["nbh"][0])
print(f"N_TARGET = {N_TARGET} (from the LH AGN run)")

In [ ]:
GENERATE = True

onep_path = f"{OUTPUT_DIR}/1p_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n{N_TARGET}.npz"

if GENERATE:
    result = run_suite(
        n_target=N_TARGET, tracer="luminosity",
        sim_path=SIM_PATH_1P, output_dir=OUTPUT_DIR,
        dir_prefix="1P_", numeric_id=False,
    )
else:
    data = np.load(onep_path, allow_pickle=True)
    result = {k: data[k] for k in ("sim_ids", "summaries", "nbh", "rgrid", "kvals")}

sim_ids = result["sim_ids"]
summaries = result["summaries"]
rgrid = result["rgrid"]
kvals = result["kvals"]
n_k, n_r = infer_layout(kvals, rgrid)

print(f"{len(sim_ids)} 1P simulations retained at N={N_TARGET}, summary shape {summaries.shape}")
print(f"dropped: {len(files_1p) - len(sim_ids)} / {len(files_1p)} (too few eligible AGN at this N)")

## 4. Trend test, per parameter

One scalar response per parameter (mean CDF at a fixed, physically-motivated
probe bin: r~4 Mpc/h, k=1 -- where the LH `Omega_m` signal peaked), tested
for a monotonic trend against the parameter's own values. Plots: always for
the 2 cosmological parameters (the LH sanity check); for astrophysics, only
the top few by significance -- 28 individual panels isn't readable, and
`trend_df` below is the actual ranked result.

In [ ]:
def _step_label(pidx, step):
    # matches the real on-disk grammar: "0" for the shared fiducial,
    # "p<N>_<M>" for positive steps, "p<N>_n<M>" for negative -- see
    # src/onep.py's module docstring
    if step == 0:
        return "0"
    return f"p{pidx}_{step}" if step > 0 else f"p{pidx}_n{abs(step)}"


probe_r = np.argmin(np.abs(rgrid - 4.0))
results_by_param = {}
trend_rows = []

for pidx, name in sorted(mapping.items()):
    steps, mean_cdfs = mean_cdf_by_step(summaries, sim_ids, param_index=pidx, n_k=n_k, n_r=n_r)

    if len(steps) == 0:
        print(f"{name}: no retained simulations at N={N_TARGET}, skipping")
        continue

    step_labels = [_step_label(pidx, s) for s in steps]
    step_values = theta_1p_all.loc[step_labels, name].values

    order = np.argsort(step_values)
    step_values, mean_cdfs = step_values[order], mean_cdfs[order]
    results_by_param[name] = (step_values, mean_cdfs)

    if len(step_values) >= 3:
        rho, p = monotonic_trend(step_values, mean_cdfs[:, 0, probe_r])
        trend_rows.append({
            "parameter": name,
            "group": "cosmological" if name in COSMO_ALIASES else "astrophysics",
            "n_steps": len(step_values),
            "spearman_rho": rho,
            "p_value": p,
        })
    else:
        print(f"{name}: only {len(step_values)} step(s) retained, too few for a trend test")

trend_df = pd.DataFrame(trend_rows)

# FDR-correct within each group separately: the 28 astrophysics parameters
# are an exploratory scan (multiple-testing risk is real -- a synthetic
# dry run of this exact pipeline turned up |rho|>=0.9, nominal p<0.05 for
# 2/28 *pure-noise* parameters purely from the n~5 sample size), while the
# 2 cosmological parameters are a single targeted check against LH's known
# result, not part of that scan.
trend_df["q_value"] = np.nan
for group, idx in trend_df.groupby("group").groups.items():
    trend_df.loc[idx, "q_value"] = benjamini_hochberg(trend_df.loc[idx, "p_value"].values)
trend_df["significant"] = trend_df["q_value"] < 0.05

trend_df = trend_df.sort_values("q_value").reset_index(drop=True)
trend_df

In [ ]:
N_TOP_ASTRO = 4  # how many astrophysics parameters to plot, by significance

for name in COSMO_ALIASES:
    if name in results_by_param:
        step_values, mean_cdfs = results_by_param[name]
        plot_1p_sweep(rgrid, kvals, step_values, mean_cdfs, name)

top_astro = trend_df.loc[trend_df["group"] == "astrophysics", "parameter"].head(N_TOP_ASTRO)
for name in top_astro:
    step_values, mean_cdfs = results_by_param[name]
    plot_1p_sweep(rgrid, kvals, step_values, mean_cdfs, name)

## Reading this

`trend_df` is a monotonicity check at one fixed, physically-motivated bin
(r~4 Mpc/h, k=1), not a full scale-resolved significance test like
`sensitivity_table`. Read `q_value`/`significant`, not the raw `p_value` --
they're FDR-corrected (Benjamini-Hochberg, as in the LH analysis) *within*
each group separately, astrophysics against astrophysics and cosmological
against cosmological, since the 28 astrophysics parameters are an
exploratory scan (a synthetic dry run of this exact pipeline turned up
`|rho|>=0.9`, nominal p<0.05, for 2 of 28 *pure-noise* parameters from the
n~5 sample size alone) while the 2 cosmological ones are a single targeted
check against LH's already-known result, not part of that scan.

Two things still temper how much weight any single row carries even after
that correction:

- **Small n.** With this release's step set (typically 4 non-fiducial steps
  plus the shared fiducial = 5 points, fewer for any parameter flagged with
  an irregular step set in section 1) and no repeated seeds per step, each
  point is a single noisy realization -- treat a significant astrophysics
  row as a lead worth a closer look (e.g. a full scale-resolved plot, or
  checking against the CV set's seed-to-seed scatter as a proper noise
  floor), not a confirmed detection at LH's standard of rigor.
- **Different questions for the two groups.** The cosmological rows are a
  direct sanity check against LH -- `Omega_m` in particular should be
  significant here too, consistent with its dominant LH signal; if it
  isn't, something upstream (N, mass cut, probe bin) needs a look before
  trusting anything else in this notebook. The astrophysics rows are
  **not** a re-test of LH's `A_SN1`/`A_AGN1`/`A_SN2`/`A_AGN2` -- they're
  this suite's own, unrelated finer decomposition of feedback physics. A
  significant astrophysics row says "this specific subgrid knob leaves a
  trace in the AGN kNN-CDF"; it says nothing about whether LH's 4 lumped
  parameters were noise-swamped, since they aren't the same
  parameterization.